In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ.setdefault('VOICE_AGENT_DRIVE', '/content/drive/MyDrive/voice-agent')
print('Drive root:', os.environ['VOICE_AGENT_DRIVE'])

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

DRIVE = Path(os.environ['VOICE_AGENT_DRIVE'])
CODE = Path('/content/runtime/code')
CODE.parent.mkdir(parents=True, exist_ok=True)

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

repo, token = _secret('GIT_REPO_URL'), _secret('GIT_REPO_TOKEN')
if repo:
    url = repo.replace('https://', f'https://x-access-token:{token}@') if token else repo
    if (CODE / '.git').exists():
        subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', '-q'], check=True)
    else:
        shutil.rmtree(CODE, ignore_errors=True)
        r = subprocess.run(['git', 'clone', '-q', url, str(CODE)], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('git clone failed (check GIT_REPO_URL / GIT_REPO_TOKEN): ' + r.stderr.replace(token or '\0', '***'))
    print('code from git:', subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())
elif (DRIVE / 'voice-agent-code.zip').exists():
    import zipfile
    shutil.rmtree(CODE, ignore_errors=True)
    with zipfile.ZipFile(DRIVE / 'voice-agent-code.zip') as zf:
        bad = [n for n in zf.namelist() if n.startswith('/') or '..' in Path(n).parts]
        if bad:
            raise ValueError(f'unsafe paths in zip: {bad[:3]}')
        zf.extractall(CODE)
    if not (CODE / 'voice_engine').exists():
        raise FileNotFoundError('voice-agent-code.zip does not contain the project (voice_engine/ missing at its top level)')
    print('code extracted from', DRIVE / 'voice-agent-code.zip')
elif (DRIVE / 'code_src' / 'voice_engine').exists():
    subprocess.run(['rsync', '-a', '--delete', '--exclude', '.git', f"{DRIVE / 'code_src'}/", f'{CODE}/'], check=True)
    print('code copied from', DRIVE / 'code_src')
else:
    raise FileNotFoundError(f"No code found: upload voice-agent-code.zip to {DRIVE}, "
                            f"or set GIT_REPO_URL, or upload the project folder to {DRIVE / 'code_src'}")

if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
# forget previously imported project modules so re-running this cell picks up new code
for _m in [m for m in sys.modules if m.split('.')[0] in ('ops', 'voice_engine', 'bench', 'gateway')]:
    del sys.modules[_m]
os.chdir(CODE)
from ops import env as ops_env
ops_env.ensure_layout()
print('persistent :', ops_env.DRIVE_ROOT)
print('ephemeral  :', ops_env.RUNTIME)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemError('No NVIDIA GPU. Runtime -> Change runtime type -> T4 GPU, then re-run from cell 1.')
print('GPU:', r.stdout.strip())
import torch
print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| cuDNN', torch.backends.cudnn.version(),
      '| available', torch.cuda.is_available())
assert torch.cuda.is_available(), 'torch cannot see the GPU'
cc = torch.cuda.get_device_capability(0)
if cc != (7, 5):
    print(f'WARNING: compute capability {cc}; the llama.cpp build in cell 12 targets sm_75 (T4). '
          'Set LLAMA_CUDA_ARCH accordingly or use a T4.')
print(subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout.splitlines()[-2])

In [ ]:
from ops.env import disk_report
rep = disk_report()
print(rep)
if rep['vm_local']['free_gb'] < 20:
    raise SystemError('Less than 20 GB free on the VM disk; restart the runtime or free space.')
if rep.get('drive', {}).get('free_gb', 99) < 20:
    print('WARNING: under 20 GB free on Google Drive; downloading every model candidate needs ~15 GB.')

In [ ]:
import hashlib, importlib, subprocess, sys
from pathlib import Path
req = CODE / 'requirements' / 'colab.txt'
marker = Path('/content/runtime') / ('.deps_' + hashlib.sha1(req.read_bytes()).hexdigest()[:12])
if sys.version_info >= (3, 13):
    print(f'NOTE: Python {sys.version.split()[0]}: the kokoro package needs < 3.13, so it will be skipped '
          '(English TTS then falls back to the next available engine). See docs.')
    reqs = [l for l in req.read_text().splitlines() if not l.startswith(('kokoro', 'misaki'))]
    req = Path('/content/runtime/tmp/requirements.txt'); req.write_text('\n'.join(reqs))
if marker.exists():
    print('dependencies already installed in this VM')
else:
    subprocess.run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq espeak-ng ffmpeg rsync lsb-release > /dev/null',
                   shell=True, check=True)
    env = dict(os.environ, PIP_CACHE_DIR=str(DRIVE / 'cache' / 'pip'))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)], check=True, env=env)
    marker.touch()

versions = {}
for mod in ['faster_whisper', 'ctranslate2', 'silero_vad', 'soundfile', 'jiwer', 'aiohttp', 'yaml',
            'transformers', 'huggingface_hub', 'torch', 'kokoro']:
    try:
        m = importlib.import_module(mod)
        versions[mod] = getattr(m, '__version__', 'ok')
    except Exception as e:
        versions[mod] = f'MISSING ({type(e).__name__}: {e})'
for k, v in versions.items():
    print(f'{k:16} {v}')
import ctranslate2
print('ctranslate2 CUDA devices:', ctranslate2.get_cuda_device_count())
from ops.env import nvidia_lib_dirs
print('CUDA wheel lib dirs passed to child processes:', len(nvidia_lib_dirs()))

In [ ]:
from ops import secrets as ops_secrets
sources = ops_secrets.load_all()
for name, src in sources.items():
    print(f'{name:28} {src:16} {ops_secrets.mask(os.environ.get(name, ""))}')
from ops.env import child_env
_e = child_env()
print('\nVOICE_AGENT_ROOT =', _e['VOICE_AGENT_ROOT'])
print('HF_HOME          =', _e['HF_HOME'])

In [ ]:
from ops import pg
from ops.services import Service
pg.install()
fresh = pg.init_cluster()
pg_svc = pg.service()
pg_svc.start(wait_s=60)
pg.bootstrap_roles()
if fresh:
    print('restore:', pg.restore_latest())
print(pg.psql('select version()'))
from ops import migrate
migrate.migrate(*migrate.colab_conn())      # tables + demo business; already-applied files are skipped
backup_svc = Service('pg-backup', [sys.executable, '-m', 'ops.pg', 'backup-loop', '600'], cwd=str(CODE))
backup_svc.start()

In [ ]:
ACTIVE_STT = 'whisper-large-v3-turbo'   #@param ['whisper-small', 'whisper-medium', 'whisper-large-v3-turbo']
ACTIVE_LLM = 'qwen3-8b'                 #@param ['qwen3-8b', 'qwen2.5-7b', 'qwen3-4b', 'gemma3-4b', 'qwen2.5-3b']
TTS_EN = 'kokoro'                       #@param ['kokoro', 'mms', 'azure']
TTS_UR = 'mms'                          #@param ['mms', 'azure']
AZURE_REGION = ''                       #@param {type:'string'}
BUSINESS_DATA = 'n8n'                   #@param ['n8n', 'static']
from ops import models as ops_models
os.environ['HF_HOME'] = str(DRIVE / 'cache' / 'hf')
if TTS_EN == 'kokoro' and not ops_env.python_ok_for_kokoro():
    TTS_EN = 'azure' if os.environ.get('AZURE_SPEECH_KEY') else 'mms'
    print(f'kokoro needs Python < 3.13 (this runtime: {sys.version.split()[0]}); English TTS ->', TTS_EN)
# download only what the agent uses (already-downloaded models are skipped)
ops_models.download('stt', [ACTIVE_STT])
ops_models.download('llm', [ACTIVE_LLM])
ops_models.download('tts', [t for t, used in (('kokoro', TTS_EN == 'kokoro'), ('mms-eng', TTS_EN == 'mms'),
                                               ('mms-urd', TTS_UR == 'mms')) if used])
cat = ops_models.catalog()
stt_dir = {m['id']: m['dir'] for m in cat['stt']}[ACTIVE_STT]
llm_meta = {m['id']: m for m in cat['llm']}[ACTIVE_LLM]
import yaml
override = yaml.safe_load((CODE / 'config' / 'voice_engine.yaml').read_text(encoding='utf-8'))
override['stt']['model_path'] = f'models/{stt_dir}'
override['tts']['engine_by_language'] = {'en': TTS_EN, 'ur': TTS_UR}
override['tts']['azure_region'] = AZURE_REGION
override['llm']['chat_template_kwargs'] = llm_meta.get('chat_template_kwargs') or {}
# 'n8n' = real menu/orders/bookings from PostgreSQL via the n8n workflows (cell 10 installs them);
# 'static' = the old fake test data in config/dev_business.yaml
override['business']['gateway'] = BUSINESS_DATA
override['business']['n8n_base_url'] = 'http://127.0.0.1:5678/webhook'
RUNTIME_CFG = Path('/content/runtime/voice_engine.runtime.yaml')
RUNTIME_CFG.write_text(yaml.safe_dump(override, allow_unicode=True, sort_keys=False), encoding='utf-8')
os.environ['VOICE_ENGINE_CONFIG'] = str(RUNTIME_CFG)   # benchmarks and child processes read this
print('runtime config ->', RUNTIME_CFG)
print({'stt': stt_dir, 'llm': llm_meta['file'], 'tts': override['tts']['engine_by_language']})

In [ ]:
from ops import llm_server
import json
llm_server.ensure_built()
print(llm_server.version()[-300:])

def start_llm(model_id, ctx=8192, parallel=2):
    meta = {m['id']: m for m in ops_models.catalog()['llm']}[model_id]
    svc = llm_server.service(ops_models.gguf_path(model_id), ctx=ctx, parallel=parallel,
                             template_kwargs=json.dumps(meta['chat_template_kwargs']) if meta.get('chat_template_kwargs') else None)
    svc.stop()
    svc.start(wait_s=300)
    return svc

llm_svc = start_llm(ACTIVE_LLM)

In [ ]:
from ops import n8n as ops_n8n
ops_n8n.ensure_node()
ops_n8n.ensure_n8n()
n8n_svc = ops_n8n.service()
n8n_svc.stop()                      # the CLI import below must run while n8n is stopped
ops_n8n.install_workflows()         # credentials (encrypted by n8n) + 7 workflows, published
n8n_svc.start(wait_s=240)
from ops.run import py
py('ops.n8n_check', check=False)    # end-to-end check against a temporary test business (PASS/FAIL per step)
from google.colab import output as colab_output
colab_output.serve_kernel_port_as_window(ops_n8n.PORT, anchor_text='Open n8n editor (Colab-authenticated)')

In [ ]:
from ops import health
_ = health.print_report()

In [ ]:
from ops import tunnel
from ops.services import http_ok
ve_svc = Service('voice-engine', [sys.executable, '-m', 'voice_engine', 'serve', '--config', str(RUNTIME_CFG)],
                 cwd=str(CODE), health=lambda: http_ok('http://127.0.0.1:8765/healthz'))
ve_svc.start(wait_s=600)
tunnel.ensure_cloudflared()
tun_svc = tunnel.service()
tun_svc.stop()                      # restart so the current settings apply (the URL changes)
tun_svc.start()
PUBLIC_URL = tunnel.public_url(tun_svc)      # waits until the tunnel is really connected
if PUBLIC_URL:
    import time
    reachable = False
    for _ in range(12):                      # new hostnames can take up to a minute to resolve
        reachable = http_ok(PUBLIC_URL + '/healthz', timeout=10)
        if reachable:
            break
        time.sleep(5)
    print('Test client:', PUBLIC_URL)
    print('public /healthz reachable:', reachable)
    print('Token: VOICE_ENGINE_WS_TOKEN from the Colab Secrets panel or Drive voice-agent/secrets/generated.env')

In [ ]:
import json, urllib.request
TEST_NUMBER = ''   #@param {type:'string'}
h = json.loads(urllib.request.urlopen('http://127.0.0.1:8765/healthz', timeout=5).read() or b'{}') if http_ok('http://127.0.0.1:8765/healthz') else {}
print('telephony:', h.get('telephony', 'voice engine not healthy'))
if TEST_NUMBER:
    req = urllib.request.Request('http://127.0.0.1:8765/calls', method='POST',
        data=json.dumps({'number': TEST_NUMBER, 'context': {'purpose': 'manual_test'}}).encode(),
        headers={'Content-Type': 'application/json', 'Authorization': 'Bearer ' + os.environ['VOICE_ENGINE_CALLS_TOKEN']})
    try:
        call = json.loads(urllib.request.urlopen(req, timeout=10).read())
        print('call requested:', call)
    except urllib.error.HTTPError as e:
        print('refused:', e.code, e.read().decode()[:200])

In [ ]:
# Follow the call (re-run while it progresses). Statuses: requested dialing alerting active ended | failed rejected
if TEST_NUMBER and 'call' in globals():
    req = urllib.request.Request(f"http://127.0.0.1:8765/calls/{call['call_id']}",
                                 headers={'Authorization': 'Bearer ' + os.environ['VOICE_ENGINE_CALLS_TOKEN']})
    print(json.loads(urllib.request.urlopen(req, timeout=5).read()))

In [ ]:
for name in ['postgres', 'pg-backup', 'llm', 'n8n', 'voice-engine', 'tunnel']:
    s = Service(name, [])
    print(f'===== {name} ({"running" if s.running() else "stopped"}) =====')
    print(s.tail(8))
turns = DRIVE / 'logs' / 'voice_engine' / 'turns.jsonl'
if turns.exists():
    import json, statistics
    rows = [json.loads(l) for l in turns.read_text(encoding='utf-8').splitlines() if l.strip()]
    for k in ['stt_s', 'llm_ttft_s', 'first_sentence_s', 'first_audio_s']:
        v = [r[k] for r in rows if r.get(k) is not None]
        if v:
            print(f'{k:18} n={len(v):4} mean={statistics.mean(v):.3f}s  max={max(v):.3f}s')

In [ ]:
print(subprocess.run('nvidia-smi --query-compute-apps=pid,process_name,used_memory --format=csv', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run('ss -ltnp | grep -E ":(5432|5678|8081|8765)\\b" || true', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run('free -h; df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)
for name in ['voice-engine', 'llm', 'n8n']:
    errs = [l for l in Service(name, []).tail(400).splitlines() if 'ERROR' in l or 'Traceback' in l]
    print(f'{name}: {len(errs)} error lines in last 400', *errs[-5:], sep='\n  ')

In [ ]:
try:
    pg.backup()
except Exception as e:                      # still stop everything; report the backup problem
    print('BACKUP FAILED:', e)
for name in ['tunnel', 'voice-engine', 'n8n', 'llm', 'pg-backup']:
    Service(name, []).stop()
pg.service().stop()
print('all services stopped; backups are in', DRIVE / 'backups' / 'pg')